In [0]:
%run /Workspace/Users/rana@ghazzi.com/Dev/Databricks_IBM/config_Parms

In [0]:
# Auto Loader: Stream reads from landing zone
stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation","/Volumes/workspace/bronze/schemas/ibm_stream")\
    .load("/Volumes/workspace/bronze/landing_zone/ibm_landing")

# Write stream to Delta table
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/bronze/checkpoints/ibm_stream") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .table(f"{catalog}.{bronze_schema}.ibm")

# Check stream status
print(query.status)

In [0]:
display(spark.table(f"{catalog}.{bronze_schema}.ibm").orderBy("date", ascending=False)) 


dbutils.fs.ls("dbfs:/Volumes/workspace/bronze/landing_zone/")